In [ ]:
# Celda 1: imports
# Este notebook reproduce el resultado de cnn_damage_classification_ETL2.ipynb
# (celdas 5c, 5e, 6 y 6b) sin los pasos de exploracion: descarga las
# imagenes de S3, aplica directamente los grupos finales ya decididos,
# limpia el dataset y guarda dataset_final.csv y grupos_final.json.
# Correr las celdas en orden, de arriba hacia abajo.

import os
import glob
import io
import json
import hashlib
import subprocess
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# Celda 2: descarga del dataset desde S3
# Crea data_clasificacion/images y data_clasificacion/labels y sincroniza imagenes, labels, classes.txt y
# data.yaml desde el bucket. Requiere tener instalado el AWS CLI y
# configuradas credenciales con acceso al bucket (aws configure).
# Si las imagenes ya estan descargadas, "aws s3 sync" solo baja lo que falta.

# todo se guarda en la carpeta data_clasificacion (se crea si no existe)
LOCAL_DATA_DIR = "data_clasificacion"
os.makedirs(f"{LOCAL_DATA_DIR}/images", exist_ok=True)
os.makedirs(f"{LOCAL_DATA_DIR}/labels", exist_ok=True)

BUCKET = "s3://humanoid-damage-cnn-834257582282-us-east-1-an/processed/damage_classification"

subprocess.run(["aws", "s3", "sync", f"{BUCKET}/images/", f"{LOCAL_DATA_DIR}/images/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "sync", f"{BUCKET}/labels/", f"{LOCAL_DATA_DIR}/labels/", "--quiet"], check=True)
subprocess.run(["aws", "s3", "cp", f"{BUCKET}/classes.txt", f"{LOCAL_DATA_DIR}/classes.txt", "--quiet"], check=True)
subprocess.run(["aws", "s3", "cp", f"{BUCKET}/data.yaml", f"{LOCAL_DATA_DIR}/data.yaml", "--quiet"], check=True)

n_images = len(os.listdir(f"{LOCAL_DATA_DIR}/images"))
n_labels = len(os.listdir(f"{LOCAL_DATA_DIR}/labels"))
print(f"Imagenes: {n_images} | Labels: {n_labels}")

In [ ]:
# Celda 3: indice de clases originales y conteo por clase
# Lee classes.txt y recorre labels/*.txt UNA sola vez para armar el indice
# {clase original: [rutas de imagen]}. Muestra cuantas imagenes tiene cada
# una de las 64 clases originales: sirve para confirmar que se descargo
# el mismo dataset (ej: Corrosion 7649, Concrete spalling 4456, Vegetation 4253).
# Las rutas se guardan con "/" para que el csv final sirva en cualquier
# sistema operativo.

with open(f"{LOCAL_DATA_DIR}/classes.txt") as f:
    all_classes = [line.strip() for line in f if line.strip()]
orig_id_to_name = {i: name for i, name in enumerate(all_classes)}

indice_clase_a_imagenes = defaultdict(list)
for label_path in glob.glob(f"{LOCAL_DATA_DIR}/labels/*.txt"):
    with open(label_path) as f:
        clases_en_imagen = {int(l.split()[0]) for l in f if l.strip()}
    if not clases_en_imagen:
        continue
    nombre_base = os.path.splitext(os.path.basename(label_path))[0]
    posibles = glob.glob(f"{LOCAL_DATA_DIR}/images/{nombre_base}.*")
    if not posibles:
        continue
    ruta_imagen = posibles[0].replace(os.sep, "/")
    for class_id in clases_en_imagen:
        indice_clase_a_imagenes[orig_id_to_name[class_id]].append(ruta_imagen)

print(f"Total de clases en el dataset original: {len(all_classes)}\n")
for name in all_classes:
    print(f"{len(indice_clase_a_imagenes.get(name, [])):>6}  {name}")

In [ ]:
# Celda 4: grupos finales (resultado de las celdas 5c y 5e del ETL2)
# GRUPOS_FINAL: superclase -> clases originales que la forman. El orden
#   de las superclases es el orden de las salidas del modelo.
# CLASES_EXCLUIDAS: descartadas a proposito (pocas imagenes o no interesan).
# CLASES_EN_DUDA: no se usan para el modelo pero no se borran del disco,
#   quedan pendientes para decidir mas adelante.
# Las clases "0" y "1" no estan en ninguna lista (sin asignar), tampoco
# se usan para el modelo.
# Al final chequea que no haya nombres mal escritos ni repetidos.

GRUPOS_FINAL = {
    "Structural crack": ["Diagonal crack", "Flexural/rocking crack", "Shear crack",
                         "Detachment crack", "stairstep_crack",
                         "-1-Earthquake-Induced Cracks And Collapses",
                         "crack", "Crack"],
    "Surface flaking/exfoliation": ["-2-Blistering", "-8-Flaking", "peeling_paint",
                                    "-9-Exfoliation", "-3-Alveolization"],
    "Salt weathering": ["-5-Salt Crystallization", "16.Subflorescence",
                        "17.Sugaring", "18.Surface Crust"],
    "Material loss/erosion": ["12.Disintegration", "14. Erosion",
                              "20. Loss Of Material-Lack"],
    "Moisture/water damage": ["water_seepage", "-7. Water Damage-Rising-Damp",
                              "Soil moisture", "Stagnant water", "Wet surface"],
    "Biological growth": ["10.Biological Colonization-Microbial",
                          "21.Patina Biologica", "mold"],
    "Corrosion": ["Corrosion", "-2.Corrosion-Local-Widespread",
                  "12-Corrosion-Local Widespread"],
    "Concrete spalling": ["Concrete spalling"],
    "Vegetation": ["Vegetation", "-4-Vegetation-Advanced Stage"],
    "Delamination": ["13-Delamination-Localized Spread",
                     "-3.Delamination-Localized-Spread"],
    "Vandalism": ["10-Vandalism"],
}

CLASES_EXCLUIDAS = [
    "Partial/full collapse", "Large displacement", "Out-of-plane mechanism",
    "15.Friability", "11. Surface Deposits", "14-Encrustation-Localized Spread",
]

CLASES_EN_DUDA = [
    "-7- Swelling", "11-Crumbling", "23.Casting", "-5-Detachment",
    "-8-Chromatic Alteration", "Honeycomb", "-6-Pitting",
    "22.Staining", "Non structural damage", "13.Efflorescence", "-1.Crumbling",
    "Void", "Debris", "damp", "stain", "Efflorescence", "Leaching",
    "Rebar deformation/buckling", "dampness",
]

FINAL_CLASSES = list(GRUPOS_FINAL)

nombres_usados = [c for subs in GRUPOS_FINAL.values() for c in subs] + CLASES_EXCLUIDAS + CLASES_EN_DUDA
mal_escritos = [c for c in nombres_usados if c not in all_classes]
repetidos = sorted({c for c in nombres_usados if nombres_usados.count(c) > 1})
sin_asignar = [c for c in all_classes if c not in set(nombres_usados)]

print("nombres que no existen en classes.txt:", mal_escritos or "ninguno")
print("clases asignadas a mas de un lugar:", repetidos or "ninguna")
print("clases sin asignar (no se usan):", sin_asignar)
print(f"\nSuperclases ({len(FINAL_CLASSES)}):")
for sup, subs in GRUPOS_FINAL.items():
    print(f"  {sup:30s} <- {len(subs)} clases originales")

In [ ]:
# Celda 5: construir el df final (una fila por imagen)
# Arma df con image_path, una columna 0/1 por superclase y la columna
# label (la superclase de la imagen, para multi-clase).
# Se descartan:
#   - imagenes que no caen en ninguna superclase (solo tienen clases
#     excluidas, en duda o sin asignar)
#   - imagenes que caen en mas de una superclase, porque en multi-clase
#     cada imagen tiene que tener una sola etiqueta
# Si una imagen tiene una clase en duda y otra asignada, se queda con la
# asignada (las clases en duda simplemente se ignoran).
# Depende de: indice_clase_a_imagenes (celda 3), GRUPOS_FINAL (celda 4)

imagen_a_superclases = defaultdict(set)
for sup, subs in GRUPOS_FINAL.items():
    for sub in subs:
        for ruta in indice_clase_a_imagenes.get(sub, []):
            imagen_a_superclases[ruta].add(sup)

todas_las_imagenes = set().union(*indice_clase_a_imagenes.values())

filas = []
for ruta in sorted(todas_las_imagenes):
    sups = imagen_a_superclases.get(ruta, set())
    fila = {"image_path": ruta, "n_superclases": len(sups)}
    for cls in FINAL_CLASSES:
        fila[cls] = int(cls in sups)
    filas.append(fila)
df_todas = pd.DataFrame(filas)

sin_superclase = df_todas[df_todas["n_superclases"] == 0]
multi_superclase = df_todas[df_todas["n_superclases"] > 1]

df = df_todas[df_todas["n_superclases"] == 1].copy()
df["label"] = df[FINAL_CLASSES].idxmax(axis=1)
df = df.drop(columns="n_superclases").reset_index(drop=True)

print(f"Imagenes con al menos una etiqueta: {len(df_todas)}")
print(f"  descartadas sin superclase:         {len(sin_superclase)}")
print(f"  descartadas con mas de una:         {len(multi_superclase)}")
print(f"  quedan (una sola superclase):       {len(df)}\n")

print("Imagenes por clase en df:")
print(df["label"].value_counts().to_string())

In [ ]:
# Celda 6: limpieza y guardado del dataset final
# Lee cada archivo UNA vez y:
#   - calcula su hash md5 (mismo hash = mismo archivo exacto)
#   - chequea que abra bien con PIL (si no, es corrupta)
#   - chequea que el formato lo pueda leer tf.io.decode_image
#     (jpeg, png, gif, bmp); otros como webp o tiff fallarian al entrenar
# Duplicados: si todas las copias tienen la misma label se deja una sola;
# si tienen labels distintas se eliminan todas (etiqueta contradictoria).
# Asi la misma foto no puede terminar en train y en test a la vez.
# Al final guarda en LOCAL_DATA_DIR:
#   - dataset_final.csv  -> columnas image_path, label
#   - grupos_final.json  -> GRUPOS_FINAL
# que son los que usa el notebook de entrenamiento multi-clase.
# Depende de: df (celda 5), GRUPOS_FINAL (celda 4), LOCAL_DATA_DIR (celda 2)

FORMATOS_OK = {"JPEG", "PNG", "GIF", "BMP"}

hashes, estados = [], []
for i, ruta in enumerate(df["image_path"]):
    with open(ruta, "rb") as f:
        contenido = f.read()
    hashes.append(hashlib.md5(contenido).hexdigest())
    try:
        with Image.open(io.BytesIO(contenido)) as img:
            formato = img.format
            img.verify()
        estados.append("ok" if formato in FORMATOS_OK else f"formato {formato}")
    except Exception:
        estados.append("corrupta")
    if (i + 1) % 5000 == 0:
        print(f"  revisadas {i + 1}/{len(df)}")

df["hash"] = hashes
df["estado"] = estados

print("\nEstado de los archivos:")
print(df["estado"].value_counts().to_string())
df_limpio = df[df["estado"] == "ok"].copy()

labels_por_hash = df_limpio.groupby("hash")["label"].nunique()
hashes_conflicto = labels_por_hash[labels_por_hash > 1].index
n_copias = df_limpio["hash"].duplicated(keep=False).sum()

n_conflicto = df_limpio["hash"].isin(hashes_conflicto).sum()
df_limpio = df_limpio[~df_limpio["hash"].isin(hashes_conflicto)]
n_antes_dedup = len(df_limpio)
df_limpio = df_limpio.drop_duplicates(subset="hash", keep="first")

print(f"\nImagenes que tienen al menos una copia exacta:     {n_copias}")
print(f"  eliminadas por copias con labels distintas:      {n_conflicto}")
print(f"  eliminadas por ser copia repetida (misma label): {n_antes_dedup - len(df_limpio)}")

df = df_limpio.drop(columns=["hash", "estado"]).reset_index(drop=True)

df[["image_path", "label"]].to_csv(f"{LOCAL_DATA_DIR}/dataset_final.csv", index=False)
with open(f"{LOCAL_DATA_DIR}/grupos_final.json", "w") as f:
    json.dump(GRUPOS_FINAL, f, indent=2, ensure_ascii=False)

print(f"\nDataset final: {len(df)} imagenes")
print(f"Guardado {LOCAL_DATA_DIR}/dataset_final.csv y {LOCAL_DATA_DIR}/grupos_final.json")

In [ ]:
# Celda 7: resumen del dataset final
# Tabla y grafico de cuantas imagenes quedaron por superclase despues de
# la limpieza. Es lo que va a ver el modelo al entrenar.
# Depende de: df, FINAL_CLASSES (celdas 4 y 6)

resumen = df["label"].value_counts().reindex(FINAL_CLASSES).fillna(0).astype(int)
resumen_df = pd.DataFrame({
    "imagenes": resumen,
    "% del dataset": (resumen / resumen.sum() * 100).round(1),
}).sort_values("imagenes", ascending=False)
print(resumen_df)

plt.figure(figsize=(10, max(4, len(FINAL_CLASSES) * 0.4)))
plt.barh(resumen_df.index[::-1], resumen_df["imagenes"][::-1])
plt.xlabel("cantidad de imagenes")
plt.title("dataset final: imagenes por superclase")
plt.tight_layout()
plt.show()